In [ ]:
!pip install albumentations pillow faiss-cpu tqdm transformers torch

In [ ]:
!pip install scikit-learn


In [18]:
import time
import cv2
import os
import numpy as np
import faiss
import torch
import pickle
import gc
from tqdm import tqdm
from PIL import Image
from transformers import ViTModel, AutoImageProcessor
import albumentations as A
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

class DinoFaceRecognition:
    def __init__(self, src_dir=None):
        self.src_dir = src_dir
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model_name = "facebook/dino-vits8"
        self.model = ViTModel.from_pretrained(self.model_name).to(self.device)
        self.model.eval()
        self.processor = AutoImageProcessor.from_pretrained(self.model_name)

        self.index = None
        self.labels = []
        self.class_to_id = {}
        self.id_to_class = {}

        self.augmentor = A.Compose([
            A.Resize(128, 128),
            A.OneOf([
                A.CoarseDropout(max_holes=1, max_height=32, max_width=32, p=1.0),
                A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            ], p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.HorizontalFlip(p=0.5)
        ])

    def augment_image(self, image, num_augments=4):
        augmented_images = []
        if image is None:
            return augmented_images

        if isinstance(image, np.ndarray):
            if len(image.shape) == 2:
                image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            elif image.shape[2] == 1:
                image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            elif image.shape[2] == 3:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            else:
                return augmented_images

        for _ in range(num_augments):
            try:
                augmented = self.augmentor(image=image)
                aug_img = augmented['image']
                if aug_img is not None and aug_img.shape[-1] == 3:
                    augmented_images.append(aug_img.astype(np.uint8))
            except:
                continue
        return augmented_images

    def load_data(self, faiss_index_path=None, metadata_path=None):
        if faiss_index_path and metadata_path:
            self.index = faiss.read_index(faiss_index_path)
            with open(metadata_path, 'rb') as f:
                metadata = pickle.load(f)
                self.labels = metadata['labels']
                self.class_to_id = metadata['label_to_id']
                self.id_to_class = {v: k for k, v in self.class_to_id.items()}

    def extract_features(self, images, normalize=True):
        try:
            if not isinstance(images, list):
                images = [images]
            valid_images = []
            for img in images:
                if img is None:
                    continue
                if isinstance(img, np.ndarray):
                    if len(img.shape) == 3 and img.shape[2] == 3:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    else:
                        continue
                elif not isinstance(img, Image.Image):
                    continue
                valid_images.append(img)

            if not valid_images:
                return None

            inputs = self.processor(images=valid_images, return_tensors="pt", padding=True).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            if normalize:
                embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
            return embeddings
        except:
            return None

    def build_and_save_faiss_index(self, features, save_path=None):
        if features is None or features.size == 0:
            raise ValueError("No features provided for FAISS index")

        features = features.astype('float16')
        if self.index is None:
            dim = features.shape[1]
            self.index = faiss.IndexFlatIP(dim)

        self.index.add(features)

        if save_path:
            faiss.write_index(self.index, save_path + '.faiss')
            with open(f"{save_path}_metadata.pkl", "wb") as f:
                pickle.dump({
                    "labels": self.labels,
                    "label_to_id": self.class_to_id
                }, f)

    def train_in_batches(self, faiss_index_path=None, num_augments=4, batch_size=10):
        if not self.src_dir:
            raise ValueError("Source directory (src_dir) not specified")

        self.labels = []
        all_class_names = sorted(os.listdir(self.src_dir))
        total_labels = len(all_class_names)

        for i in tqdm(range(0, total_labels, batch_size)):
            batch_class_names = all_class_names[i:i + batch_size]
            print(f"[Batch {i // batch_size + 1}] Training on classes: {batch_class_names} ({i + len(batch_class_names)} / {total_labels})")

            images = []
            labels = []

            for class_name in batch_class_names:
                class_dir = os.path.join(self.src_dir, class_name)
                if os.path.isdir(class_dir):
                    for img_name in os.listdir(class_dir):
                        img_path = os.path.join(class_dir, img_name)
                        if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                            img = cv2.imread(img_path)
                            if img is None:
                                continue
                            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                            images.append(img)
                            labels.append(class_name)
                            augmented_imgs = self.augment_image(img, num_augments=num_augments)
                            images.extend(augmented_imgs)
                            labels.extend([class_name] * num_augments)

            if not images:
                continue

            for label in set(labels):
                if label not in self.class_to_id:
                    new_id = len(self.class_to_id)
                    self.class_to_id[label] = new_id
                    self.id_to_class[new_id] = label

            batch_label_ids = [self.class_to_id[label] for label in labels]
            self.labels.extend(batch_label_ids)

            features = self.extract_features(images)
            if features is None:
                continue

            self.build_and_save_faiss_index(features, save_path=faiss_index_path)
            gc.collect()
            torch.cuda.empty_cache()

    def recognize_face_topk(self, query_img, threshold=0.5, top_k=5):
        if query_img is None or query_img.size == 0:
            return [("Invalid Image", 0.0)]
        try:
            query_img = cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB)
            query_embed = self.extract_features([query_img])
            if query_embed is None:
                return [("No Features Extracted", 0.0)]
    
            similarities, indices = self.index.search(query_embed.astype('float16'), k=top_k)
    
            votes = {}
            for i in range(top_k):
                pred_id = self.labels[indices[0][i]]
                label = self.id_to_class.get(pred_id, "Unknown")
                sim = float(similarities[0][i])
                if label not in votes:
                    votes[label] = []
                votes[label].append(sim)
    
            # Tính tổng điểm trung bình cho mỗi label
            label_scores = {label: np.mean(sims) for label, sims in votes.items()}
            best_label = max(label_scores, key=label_scores.get)
            best_score = label_scores[best_label]
    
            if best_score >= threshold:
                return [(best_label, best_score)]
            else:
                return [("Unknown", best_score)]
        except Exception as e:
            print(f"[ERROR] in recognize_face_topk: {e}")
            return [("Error", 0.0)]


    def print_faiss_size(self, faiss_file_path):
        if os.path.exists(faiss_file_path):
            size_in_bytes = os.path.getsize(faiss_file_path)
            size_in_mb = size_in_bytes / (1024 * 1024)
            print(f"[INFO] FAISS Index size: {size_in_mb:.2f} MB ({size_in_bytes} bytes)")
        else:
            print(f"[WARN] FAISS file not found: {faiss_file_path}")

    def evaluate_on_testset(self, test_dir, threshold=0.5, top_k=5):
        true_labels = []
        predicted_labels = []
        scores = []

        class_names = sorted(os.listdir(test_dir))
        for label in class_names:
            class_path = os.path.join(test_dir, label)
            if not os.path.isdir(class_path):
                continue
            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(class_path, img_name)
                    img = cv2.imread(img_path)
                    if img is None:
                        continue
                    result = self.recognize_face_topk(img, threshold=threshold, top_k=top_k)[0]
                    pred_label, score = result
                    true_labels.append(label)
                    print(f'pred_label: {pred_label} | true_label: {label} | score: {score}')
                    predicted_labels.append(pred_label)
                    scores.append(score)

        acc = accuracy_score(true_labels, predicted_labels)
        f1 = f1_score(true_labels, predicted_labels, average='macro')

        label_list = sorted(set(true_labels + predicted_labels))
        label_to_idx = {label: idx for idx, label in enumerate(label_list)}
        y_true = [label_to_idx[l] for l in true_labels]
        y_score_matrix = np.zeros((len(scores), len(label_list)))
        for i, (pl, sc) in enumerate(zip(predicted_labels, scores)):
            if pl in label_to_idx:
                y_score_matrix[i][label_to_idx[pl]] = sc

        try:
            roc_auc = roc_auc_score(y_true, y_score_matrix, multi_class='ovr')
        except:
            roc_auc = "Cannot compute ROC AUC (possibly not enough classes)"

        return {
            "Accuracy": acc,
            "F1 Score (macro)": f1,
            "ROC AUC": roc_auc
        }



In [19]:
dino = DinoFaceRecognition(src_dir='train')
# 🆕 Train with augmentation
#dino.train_in_batches(faiss_index_path='faiss_index', num_augments=4, batch_size=1)

Some weights of ViTModel were not initialized from the model checkpoint at facebook/dino-vits8 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
dino.print_faiss_size('faiss_index_no_augment.faiss')
#|%%--%%| <AGMxKnSMeM|mJYsVXolyl>
if dino.index is not None:
    print(f"[INFO] FAISS Index contains {dino.index.ntotal} vectors.")
else:
    print("[WARN] FAISS index not initialized.")

[INFO] FAISS Index size: 174.59 MB (183075885 bytes)
[INFO] FAISS Index contains 119190 vectors.


In [20]:
dino.load_data('faiss_index_no_augment.faiss', 'faiss_index_no_augment_metadata.pkl')

In [21]:
img = cv2.imread('test/n001299/0006_01.jpg')
dino.recognize_face_topk(img)

[('n004237', np.float64(0.8204189538955688))]

In [22]:
results = dino.evaluate_on_testset(test_dir='test')
print(results)

pred_label: n001299 | true_label: n001299 | score: 0.9103913307189941
pred_label: n001299 | true_label: n001299 | score: 0.9999936819076538
pred_label: n004237 | true_label: n001299 | score: 0.8204189538955688
pred_label: n001299 | true_label: n001299 | score: 0.8937224745750427
pred_label: n001299 | true_label: n001299 | score: 0.8960888385772705
pred_label: n001299 | true_label: n001299 | score: 0.8420474529266357
pred_label: n001299 | true_label: n001299 | score: 0.8217582702636719
pred_label: n001299 | true_label: n001299 | score: 1.0000882148742676
pred_label: n001299 | true_label: n001299 | score: 0.7892552614212036
pred_label: n007594 | true_label: n001299 | score: 0.7425269484519958
pred_label: n004338 | true_label: n001299 | score: 0.6771526336669922
pred_label: n001299 | true_label: n001299 | score: 0.7504342079162598
pred_label: n001299 | true_label: n001299 | score: 0.834315299987793
pred_label: n001299 | true_label: n001299 | score: 0.8771498203277588
pred_label: n007210 |